# Fine-tuning e regiões utilizadas por uma CNN

Notebook independente para o artigo. Ele não usa `experiment.py` nem outros arquivos do projeto: baixa o Oxford-IIIT Pet, treina a ResNet-18, calcula as métricas e gera os mapas Grad-CAM.

Execute as células em ordem. A etapa principal é controlada por `RUN_FULL`; deixe-a como `False` na primeira execução e use o teste rápido antes.

In [ ]:
!pip -q install 'torch>=2.2' 'torchvision>=0.17' matplotlib pandas pillow

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv, json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch import nn
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import InterpolationMode

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('/content/cnn_data')
RESULTS_DIR = Path('/content/cnn_results')
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
STRATEGIES = ('frozen', 'full')
print('Dispositivo:', DEVICE)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def stratified_split(labels, validation_fraction=0.20, train_fraction=1.0, seed=42):
    grouped = defaultdict(list)
    for index, label in enumerate(labels): grouped[int(label)].append(index)
    rng = random.Random(seed); train_indices=[]; validation_indices=[]
    for indices in grouped.values():
        rng.shuffle(indices); n_val=max(1,round(len(indices)*validation_fraction))
        validation_indices.extend(indices[:n_val]); candidates=indices[n_val:]
        train_indices.extend(candidates[:max(1,round(len(candidates)*train_fraction))])
    rng.shuffle(train_indices); rng.shuffle(validation_indices)
    return train_indices, validation_indices

class PetEvaluationDataset(Dataset):
    def __init__(self, root):
        self.dataset=OxfordIIITPet(root=root,split='test',target_types=('category','segmentation'),download=True)
        self.image_transform=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
        self.mask_transform=transforms.Compose([transforms.Resize(256,interpolation=InterpolationMode.NEAREST),transforms.CenterCrop(224),transforms.PILToTensor()])
    def __len__(self): return len(self.dataset)
    def __getitem__(self,index):
        image,target=self.dataset[index]; label,trimap=target; mask=self.mask_transform(trimap).squeeze(0)
        return self.image_transform(image),int(label),(mask!=2)

def make_loaders(seed=42,train_fraction=1.0,test_limit=0,batch_size=64,workers=0):
    train_transform=transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.75,1.0)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.10),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    eval_transform=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    metadata=OxfordIIITPet(root=DATA_DIR,split='trainval',target_types='category',download=True)
    labels=getattr(metadata,'_labels',None)
    if labels is None: labels=[metadata[i][1] for i in range(len(metadata))]
    train_indices,val_indices=stratified_split(labels,seed=seed,train_fraction=train_fraction)
    train_data=OxfordIIITPet(root=DATA_DIR,split='trainval',target_types='category',transform=train_transform,download=False)
    val_data=OxfordIIITPet(root=DATA_DIR,split='trainval',target_types='category',transform=eval_transform,download=False)
    test_data=PetEvaluationDataset(DATA_DIR)
    if test_limit: test_data=Subset(test_data,range(min(test_limit,len(test_data))))
    options=dict(batch_size=batch_size,num_workers=workers,pin_memory=torch.cuda.is_available())
    generator=torch.Generator().manual_seed(seed)
    train_loader=DataLoader(Subset(train_data,train_indices),shuffle=True,generator=generator,**options)
    val_loader=DataLoader(Subset(val_data,val_indices),shuffle=False,**options)
    test_loader=DataLoader(test_data,shuffle=False,**options)
    return train_loader,val_loader,test_loader,metadata.classes
